# 1. Setup & Dependencies
Import necessary libraries for building the model from scratch.


In [14]:
import os
import tensorflow as tf
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import pandas as pd
import time
import tensorflow as tf
from tensorflow.keras import regularizers
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
import numpy as np
from tqdm import tqdm

# Ensure reproducible results
tf.random.set_seed(42)
np.random.seed(42)

print(f"TensorFlow Version: {tf.__version__}")

# Check for GPU availability
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"Number of GPUs available: {len(gpus)}")
    for gpu in gpus:
        print(f"  GPU Name: {gpu.name}")
        # Optional: Print more details like memory limit if set
        # tf.config.experimental.set_memory_growth(gpu, True)
else:
    print("No GPU devices found. TensorFlow will run on CPU.")

TensorFlow Version: 2.20.0
Number of GPUs available: 1
  GPU Name: /physical_device:GPU:0


# 2. Data Pipeline
- Function to load and preprocess images.
- Creating `tf.data.Dataset` for training and validation.
- Implementing manual data augmentation using `tf.image` functions.


In [2]:
import os
import pandas as pd

# Check if running in Google Colab
try:
    import google.colab
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Running in Google Colab. Mounting Google Drive...")
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/dermind'
else:
    print("Running on a local device.")
    BASE_DIR = '.'

# Path configuration for manifest files
DERMNET_MANIFEST_PATH = os.path.join(BASE_DIR, 'data', 'dermnet', 'dermnet_manifest_cleaned.csv')
ACNE_MANIFEST_PATH = os.path.join(BASE_DIR, 'data', 'acne_dataset_manifest_cleaned.csv')

try:
    df_gdrive = pd.read_csv(DERMNET_MANIFEST_PATH)
    df_gdrive2 = pd.read_csv(ACNE_MANIFEST_PATH)
    print('Datasets loaded successfully:')
    print(f'  DermNet manifest shape: {df_gdrive.shape}')
    print(f'  Acne manifest shape:    {df_gdrive2.shape}')
    display(df_gdrive.head())
except FileNotFoundError:
    print(f"Error: Manifest files not found. Please ensure they exist at:\n  - {DERMNET_MANIFEST_PATH}\n  - {ACNE_MANIFEST_PATH}")
except Exception as e:
    print(f"An error occurred while loading manifests: {e}")

Mounted at /content/drive
Dataset loaded successfully from Google Drive:


,filename,filepath,label,split,file_size_bytes,format
0,eczema-herpeticum-46.jpg,train/Warts Molluscum and other Viral Infectio...,Warts Molluscum and other Viral Infections,train,107644,JPG
1,eczema-herpeticum-56.jpg,train/Warts Molluscum and other Viral Infectio...,Warts Molluscum and other Viral Infections,train,91593,JPG
2,eczema-herpeticum-35.jpg,train/Warts Molluscum and other Viral Infectio...,Warts Molluscum and other Viral Infections,train,73979,JPG
3,eczema-herpeticum-55.jpg,train/Warts Molluscum and other Viral Infectio...,Warts Molluscum and other Viral Infections,train,69588,JPG
4,eczema-herpeticum-61.jpg,train/Warts Molluscum and other Viral Infectio...,Warts Molluscum and other Viral Infections,train,88420,JPG


In [3]:
if IN_COLAB:
    print("Copying and extracting dataset in Colab...")
    # 2. Copy the zip file to Colab's fast local SSD storage
    !cp /content/drive/MyDrive/dermind/dermnet.zip /content/

    # 3. Unzip it silently into Colab's local space
    !unzip -q /content/dermnet.zip -d /content/dataset/
else:
    print("Running locally. Skipping dataset zip copy and extraction (using local data folder).")

In [19]:
# Configuration parameters
BATCH_SIZE = 32
IMG_SIZE = (299, 299)

# Set base dataset paths
if IN_COLAB:
    GDRIVE_BASE_PATH = "/content/dataset/"
else:
    GDRIVE_BASE_PATH = "./data/"

# 1. Load DermNet manifest
df_derm = df_gdrive.copy()

# Set full paths for DermNet images
if IN_COLAB:
    df_derm['full_path'] = df_derm['filepath'].apply(lambda x: os.path.join(GDRIVE_BASE_PATH, x))
else:
    # Locally, filepath is relative to the 'dermnet' directory (e.g. train/...)
    df_derm['full_path'] = df_derm['filepath'].apply(lambda x: os.path.join(GDRIVE_BASE_PATH, 'dermnet', x))

# Filter out histology images from DermNet
df_derm = df_derm[~df_derm['filename'].str.contains('histology', case=False, na=False)].copy()

# 2. Load Acne dataset manifest
df_acne = df_gdrive2.copy()
df_acne['label'] = 'Acne and Rosacea Photos'  # Map to DermNet's acne label temporarily

# Set full paths for Acne images
if IN_COLAB:
    df_acne['full_path'] = df_acne['filepath'].apply(lambda x: os.path.join(GDRIVE_BASE_PATH, x))
else:
    # Locally, Acne images are placed under data/dermnet/train/Acne and Rosacea Photos or data/dermnet/test/Acne and Rosacea Photos
    def get_local_acne_path(row):
        filename = row['filename']
        # Try train path
        path_train = os.path.join(GDRIVE_BASE_PATH, 'dermnet', 'train', 'Acne and Rosacea Photos', filename)
        if os.path.exists(path_train):
            return path_train
        # Try test path
        path_test = os.path.join(GDRIVE_BASE_PATH, 'dermnet', 'test', 'Acne and Rosacea Photos', filename)
        if os.path.exists(path_test):
            return path_test
        # Fallback to the default path in manifest
        return os.path.join(GDRIVE_BASE_PATH, row['filepath'])

    df_acne['full_path'] = df_acne.apply(get_local_acne_path, axis=1)

# Filter out histology images from Acne
df_acne = df_acne[~df_acne['filename'].str.contains('histology', case=False, na=False)].copy()

Found 3 classes.
Building training dataset...
Building validation dataset...
Building test dataset...
Sample batch images shape: (32, 299, 299, 3)
Sample batch labels shape: (32, 3)


In [20]:
# ── Distribution of images per class and per split ──

# Cross-tabulate: rows = class labels, columns = splits
dist = pd.crosstab(df_merged['label'], df_merged['split'], margins=True, margins_name='Total')

# Sort by Total (descending) for easier reading, keep 'Total' row at the bottom
dist_sorted = dist.iloc[:-1].sort_values('Total', ascending=False)
dist_sorted = pd.concat([dist_sorted, dist.iloc[[-1]]])

print('=' * 65)
print('  Image Distribution per Class and Split')
print('=' * 65)
print(dist_sorted.to_string())
print('=' * 65)


  Image Distribution per Class and Split
split                test  train  Total
label                                  
Infections           1089   4354   5443
Eczema & Dermatitis   544   2178   2722
Acne & Rosacea        172    686    858
Total                1805   7218   9023


# 3. Custom Architecture
- Implement a `ResBlock` using Layer Subclassing.
- Implement the `SkinClassifierNet` using Model Subclassing.


In [21]:
class SqueezeExcitation(tf.keras.layers.Layer):
    def __init__(self, channels, reduction=16, **kwargs):
        super(SqueezeExcitation, self).__init__(**kwargs)
        self.channels = channels
        self.reduction = reduction

        # Ensure the bottleneck has at least 1 channel
        bottleneck_channels = max(1, channels // reduction)

        self.global_pool = tf.keras.layers.GlobalAveragePooling2D()

        # Added weight decay to match the rest of your network
        self.fc1 = tf.keras.layers.Dense(
            bottleneck_channels,
            activation='relu',
            kernel_regularizer=regularizers.l2(1e-4)
        )
        self.fc2 = tf.keras.layers.Dense(
            channels,
            activation='sigmoid',
            kernel_regularizer=regularizers.l2(1e-4)
        )

    def call(self, inputs):
        # Squeeze: [batch, height, width, channels] -> [batch, channels]
        x = self.global_pool(inputs)
        x = self.fc1(x)
        x = self.fc2(x)

        # Excite: Reshape to permit broadcasting.
        # Using tf.expand_dims is safer and more robust than hardcoding [-1, 1, 1, ...]
        x = tf.expand_dims(tf.expand_dims(x, axis=1), axis=1) # [batch, 1, 1, channels]

        return inputs * x

In [22]:
class ResBlock(tf.keras.layers.Layer):
    """
    A Custom Residual Block with two Conv2D layers, Squeeze-and-Excitation attention,
    and an optional shortcut connection.
    """
    def __init__(self, filters, stride=1, use_se=True, **kwargs):
        super(ResBlock, self).__init__(**kwargs)
        self.filters = filters
        self.stride = stride
        self.use_se = use_se

        # Main computation path
        self.conv1 = tf.keras.layers.Conv2D(filters, kernel_size=3, strides=stride, padding='same', use_bias=False, kernel_regularizer=regularizers.l2(1e-4))
        self.bn1 = tf.keras.layers.BatchNormalization()

        self.conv2 = tf.keras.layers.Conv2D(filters, kernel_size=3, strides=1, padding='same', use_bias=False, kernel_regularizer=regularizers.l2(1e-4))
        self.bn2 = tf.keras.layers.BatchNormalization()

        # Instantiate SE Attention
        if self.use_se:
            self.se = SqueezeExcitation(filters)

        self.use_shortcut = False
        self.shortcut_conv = None
        self.shortcut_bn = None

    def build(self, input_shape):
        # Handle channel mismatch or stride
        if self.stride != 1 or input_shape[-1] != self.filters:
            self.use_shortcut = True
            self.shortcut_conv = tf.keras.layers.Conv2D(self.filters, kernel_size=1, strides=self.stride, use_bias=False, kernel_regularizer=regularizers.l2(1e-4))
            self.shortcut_bn = tf.keras.layers.BatchNormalization()
        super(ResBlock, self).build(input_shape)

    def call(self, inputs, training=False):
        # Pass through main layers
        x = self.conv1(inputs)
        x = self.bn1(x, training=training)
        x = tf.nn.gelu(x)

        x = self.conv2(x)
        x = self.bn2(x, training=training)

        # Apply SE Attention right before shortcut addition
        if self.use_se:
            x = self.se(x)

        # Pass input through shortcut
        if self.use_shortcut:
            shortcut_x = self.shortcut_conv(inputs)
            shortcut_x = self.shortcut_bn(shortcut_x, training=training)
        else:
            shortcut_x = inputs

        # Add skip connection
        x = tf.keras.layers.add([x, shortcut_x])
        return tf.nn.gelu(x)

In [23]:
class SkinClassifierNet(tf.keras.Model):
    """
    The main Custom CNN Classifier Model using Residual Blocks.
    """
    def __init__(self, num_classes, **kwargs):
        super(SkinClassifierNet, self).__init__(**kwargs)

        # Less aggressive stem: 3x3 conv, stride 2, no immediate max pool
        self.conv1 = tf.keras.layers.Conv2D(64, kernel_size=3, strides=2, padding='same', use_bias=False, kernel_regularizer=regularizers.l2(1e-4))
        self.bn1 = tf.keras.layers.BatchNormalization()

        self.blocks = tf.keras.Sequential([
            ResBlock(64, stride=1),
            ResBlock(64, stride=1),
            ResBlock(128, stride=2),
            ResBlock(128, stride=1),
            ResBlock(256, stride=2),
            ResBlock(256, stride=1),
            ResBlock(512, stride=2),
            ResBlock(512, stride=1),
        ])

        # Classifier head
        self.global_pool = tf.keras.layers.GlobalAveragePooling2D()
        self.dropout = tf.keras.layers.Dropout(0.5)
        self.classifier = tf.keras.layers.Dense(num_classes, activation='softmax', kernel_regularizer=regularizers.l2(1e-4))

    def call(self, inputs, training=False):
        x = self.conv1(inputs)
        x = self.bn1(x, training=training)
        x = tf.nn.gelu(x)

        x = self.blocks(x, training=training)

        x = self.global_pool(x)
        x = self.dropout(x, training=training)
        return self.classifier(x)

# Instantiate the model
NUM_CLASSES = num_classes  # Automatically determined from dataset
model = SkinClassifierNet(num_classes=NUM_CLASSES)

# Build the model and print the summary using a dummy input
dummy_input = tf.zeros((1, IMG_SIZE[0], IMG_SIZE[1], 3))
model(dummy_input)
model.summary()

Model: "skin_classifier_net_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_20 (Conv2D)              │ (1, 150, 150, 64)      │         1,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_20          │ (1, 150, 150, 64)      │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_1 (Sequential)       │ (1, 19, 19, 512)       │    11,265,528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_17     │ ?                      │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_33 (Dense)                │ (1, 3)                 │         1,539 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,269,051 (42.99 MB)

 Trainable params: 11,259,451 (42.95 MB)

 Non-trainable params: 9,600 (37.50 KB)

# 4. Loss and Optimizer Setup
Define the `CategoricalCrossentropy` loss and `Adam` optimizer.


In [24]:
EPOCHS = 80

# Get the number of training samples (assuming train_ds is created from df_merged and filtered for 'train' split)
num_train_samples = len(df_merged[df_merged['split'] == 'train'])

# Calculate steps per epoch
# tf.data.Dataset.cardinality() would be more robust for dynamic datasets, but len(df) / BATCH_SIZE is fine here.
steps_per_epoch = tf.math.ceil(num_train_samples / BATCH_SIZE).numpy().astype(int)

# EPOCHS is defined in cell bc90af32, ensure it's run before this cell
total_train_steps = EPOCHS * steps_per_epoch

print(f"Number of training samples: {num_train_samples}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Steps per epoch: {steps_per_epoch}")
print(f"Total training steps (for {EPOCHS} epochs): {total_train_steps}")

Number of training samples: 7218
Batch size: 32
Steps per epoch: 226
Total training steps (for 80 epochs): 18080


In [25]:
                                  # Define Loss Function for one-hot encoded multi-class problem (added label smoothing)
loss_fn = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1)

# Define Adam Optimizer
lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=3e-4,
    decay_steps=total_train_steps,
    alpha=1e-6
)

optimizer = tf.keras.optimizers.AdamW(
    learning_rate=lr_schedule,
    weight_decay=1e-4
)

# Define Metrics to track Training and Validation performance
train_loss_metric = tf.keras.metrics.Mean(name='train_loss')
train_acc_metric = tf.keras.metrics.CategoricalAccuracy(name='train_accuracy')

val_loss_metric = tf.keras.metrics.Mean(name='val_loss')
val_acc_metric = tf.keras.metrics.CategoricalAccuracy(name='val_accuracy')

# Extract training labels for class weight calculation
train_labels = df_merged[df_merged['split'] == 'train']['label'].map(class_to_idx).values

class_names_for_weights = np.unique(train_labels)

weights = compute_class_weight(
    class_weight='balanced',
    classes=class_names_for_weights,
    y=train_labels
)

class_weights = tf.constant(weights, dtype=tf.float32)

# 5. Custom Training Step
A `@tf.function`-decorated train step to calculate and apply gradients using `tf.GradientTape`.


In [26]:
@tf.function
def train_step(images, labels):
    """
    Executes one training step on a batch of data.
    """
    with tf.GradientTape() as tape:
        # 1. Forward pass: compute predictions
        # training=True is necessary for BatchNorm and Dropout layers
        predictions = model(images, training=True)

        # Convert one-hot encoded labels to integer indices for tf.gather
        integer_labels = tf.argmax(labels, axis=-1, output_type=tf.int32)

        # 2. Compute loss
        sample_weights = tf.gather(class_weights, integer_labels)

        loss = loss_fn(
          labels,
          predictions,
          sample_weight=sample_weights
        )
        # Removed redundant loss calculation: loss = loss_fn(labels, predictions)

    # 3. Calculate gradients with respect to trainable variables
    gradients = tape.gradient(loss, model.trainable_variables)

    # 4. Apply gradients to update weights
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))

    # 5. Update state for metrics
    train_loss_metric(loss)
    train_acc_metric(labels, predictions)

    return loss

# 6. Custom Validation Step
Calculates validation loss and metrics without tracking gradients.


In [27]:
@tf.function
def val_step(images, labels):
    """
    Executes one validation step on a batch of data.
    """
    # 1. Forward pass: compute predictions (training=False)
    predictions = model(images, training=False)

    # 2. Compute loss
    loss = loss_fn(labels, predictions)

    # 3. Update state for metrics
    val_loss_metric(loss)
    val_acc_metric(labels, predictions)

    return loss


# 7. The Training Loop
Runs training for `N` epochs, updates metrics, prints results, and tracks the best model weights.


In [ ]:
best_val_loss = float('inf')

history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': []
}

print("Starting custom training loop...")

for epoch in range(EPOCHS):
    start_time = time.time()

    # Reset metrics
    train_loss_metric.reset_state()
    train_acc_metric.reset_state()
    val_loss_metric.reset_state()
    val_acc_metric.reset_state()

    # =========================
    # TRAINING LOOP
    # =========================
    train_bar = tqdm(
        train_ds,
        desc=f"Epoch {epoch+1}/{EPOCHS} [Training]",
        leave=False
    )

    for step, (x_batch_train, y_batch_train) in enumerate(train_bar):

        train_step(x_batch_train, y_batch_train)

        # Update progress bar live
        train_bar.set_postfix({
            "loss": f"{train_loss_metric.result().numpy():.4f}",
            "acc": f"{train_acc_metric.result().numpy():.4f}"
        })

    # =========================
    # VALIDATION LOOP
    # =========================
    val_bar = tqdm(
        val_ds,
        desc=f"Epoch {epoch+1}/{EPOCHS} [Validation]",
        leave=False
    )

    for x_batch_val, y_batch_val in val_bar:

        val_step(x_batch_val, y_batch_val)

        # Update validation metrics live
        val_bar.set_postfix({
            "val_loss": f"{val_loss_metric.result().numpy():.4f}",
            "val_acc": f"{val_acc_metric.result().numpy():.4f}"
        })

    # Final epoch metrics
    t_loss = train_loss_metric.result().numpy()
    t_acc = train_acc_metric.result().numpy()
    v_loss = val_loss_metric.result().numpy()
    v_acc = val_acc_metric.result().numpy()

    # Save history
    history['train_loss'].append(t_loss)
    history['train_acc'].append(t_acc)
    history['val_loss'].append(v_loss)
    history['val_acc'].append(v_acc)

    # Save best model
    if v_loss < best_val_loss:
        best_val_loss = v_loss
        weights_save_path = 'best_skin_classifier_weights.h5' if not IN_COLAB else '/content/drive/MyDrive/dermind/best_skin_classifier.weights.h5'
        model.save_weights(weights_save_path)
        saved_msg = f" [New best model weights saved to {weights_save_path}]"
    else:
        saved_msg = ""

    epoch_time = time.time() - start_time

    print(
        f"Epoch {epoch+1}/{EPOCHS} ({epoch_time:.2f}s)\n"
        f"Train Loss: {t_loss:.4f} | Train Acc: {t_acc:.4f} || "
        f"Val Loss: {v_loss:.4f} | Val Acc: {v_acc:.4f}{saved_msg}"
    )

print("Training completed.")

Starting custom training loop...


Epoch 1/80 (292.28s)
Train Loss: 1.1703 | Train Acc: 0.3691 || Val Loss: 1.3848 | Val Acc: 0.0953 [New best model weights]


Epoch 2/80 (258.56s)
Train Loss: 1.1163 | Train Acc: 0.3692 || Val Loss: 1.1740 | Val Acc: 0.2006 [New best model weights]


Epoch 3/80 (254.07s)
Train Loss: 1.0822 | Train Acc: 0.4034 || Val Loss: 1.0899 | Val Acc: 0.3906 [New best model weights]


Epoch 4/80 (254.88s)
Train Loss: 1.0661 | Train Acc: 0.4095 || Val Loss: 1.0313 | Val Acc: 0.4499 [New best model weights]


...

Epoch 32/80 (252.28s)
Train Loss: 0.8796 | Train Acc: 0.5858 || Val Loss: 0.9126 | Val Acc: 0.5839


Epoch 33/80 (251.90s)
Train Loss: 0.8637 | Train Acc: 0.5907 || Val Loss: 0.8715 | Val Acc: 0.6122


Epoch 34/80 (252.20s)
Train Loss: 0.8548 | Train Acc: 0.5998 || Val Loss: 0.9312 | Val Acc: 0.5767


Epoch 35/80 (253.85s)
Train Loss: 0.8526 | Train Acc: 0.6021 || Val Loss: 0.7893 | Val Acc: 0.6892 [New best model weights]


Epoch 36/80 [Training]:  98%|█████████▊| 222/226 [03:52<00:04,  1.05s/it, loss=0.8483, acc=0.6018]

# 8. Evaluation & Visualization
Plots the learning curves and shows mock evaluation inferences.


In [ ]:
# Plot Training vs Validation Learning Curves
epochs_range = range(1, EPOCHS + 1)

plt.figure(figsize=(14, 5))

# Plot Loss
plt.subplot(1, 2, 1)
plt.plot(epochs_range, history['train_loss'], label='Training Loss', marker='o', linewidth=2)
plt.plot(epochs_range, history['val_loss'], label='Validation Loss', marker='o', linewidth=2)
plt.title('Training and Validation Loss', fontsize=14)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Categorical Crossentropy Loss', fontsize=12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

# Plot Accuracy
plt.subplot(1, 2, 2)
plt.plot(epochs_range, history['train_acc'], label='Training Accuracy', marker='s', linewidth=2)
plt.plot(epochs_range, history['val_acc'], label='Validation Accuracy', marker='s', linewidth=2)
plt.title('Training and Validation Accuracy', fontsize=14)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Categorical Accuracy', fontsize=12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

# Run Mock Evaluation
print("--- EVALUATION SUMMARY ---")
# Simulate loading the best weights
# weights_save_path = 'best_skin_classifier_weights.h5' if not IN_COLAB else '/content/drive/MyDrive/dermind/best_skin_classifier.weights.h5'\n# model.load_weights(weights_save_path)

final_val_loss = history['val_loss'][-1]
final_val_acc = history['val_acc'][-1]
print(f"Final best tracked validation loss: {best_val_loss:.4f}")

print("\nMock inference on a single batch from the validation dataset:")
# Grab one batch
sample_images, sample_labels = next(iter(val_ds))

# Inference (using training=False)
predictions = model(sample_images, training=False)

# Convert predictions and labels from probabilities/one-hot to class indices
predicted_classes = tf.argmax(predictions, axis=1)
true_classes = tf.argmax(sample_labels, axis=1)

print(f"True class indices:      {true_classes.numpy()[:10]}")
print(f"Predicted class indices: {predicted_classes.numpy()[:10]}")


In [ ]:
# Evaluate on the test set
print("--- TEST SET EVALUATION ---")

if test_ds is not None:
    test_loss_metric = tf.keras.metrics.Mean(name='test_loss')
    test_acc_metric = tf.keras.metrics.CategoricalAccuracy(name='test_accuracy')

    for x_batch_test, y_batch_test in test_ds:
        predictions = model(x_batch_test, training=False)
        loss = loss_fn(y_batch_test, predictions)
        test_loss_metric(loss)
        test_acc_metric(y_batch_test, predictions)

    print(f"Test Loss: {test_loss_metric.result().numpy():.4f}")
    print(f"Test Accuracy: {test_acc_metric.result().numpy():.4f}")


## Confusion Matrix
Compute and visualize the confusion matrix on the test set using `sklearn`.


In [ ]:
# ── Collect all predictions and true labels from the test set ──
all_true_labels = []
all_pred_labels = []

for x_batch, y_batch in test_ds:
    preds = model(x_batch, training=False)
    all_true_labels.append(tf.argmax(y_batch, axis=1).numpy())
    all_pred_labels.append(tf.argmax(preds, axis=1).numpy())

all_true_labels = np.concatenate(all_true_labels)
all_pred_labels = np.concatenate(all_pred_labels)

# ── Compute confusion matrix ──
cm = confusion_matrix(all_true_labels, all_pred_labels, labels=range(num_classes))

# ── Plot confusion matrix ──
fig, ax = plt.subplots(figsize=(22, 20))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(ax=ax, cmap='Blues', xticks_rotation=90, values_format='d')
ax.set_title('Confusion Matrix — Test Set', fontsize=18, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Classification Report ──
print('\n' + '=' * 65)
print('  Classification Report (Test Set)')
print('=' * 65)
print(classification_report(
    all_true_labels,
    all_pred_labels,
    target_names=class_names,
    zero_division=0
))
print('=' * 65)
